In [23]:
%reload_ext autoreload
%autoreload 2

In [24]:
import sys
import numpy as np

path_pjt = (
    "/Users/Nguye071/Documents/GitHub/Single-and-Multi-level-Fourier-RQMC-for-MSRM"
)
sys.path.insert(0, path_pjt)
sys.path.append(path_pjt + "/10D Gauss QPC Loss")
from QMC_MN_QPC import RQMC_Fou_MN_qpc
from QMC_CV_MN_QPC import RQMC_CV_Fou_MN_qpc
from MC_loss import MCLossFunction
from scipy.optimize import minimize


# 10D case

In [ ]:
maxiter = 3500
mu = np.zeros(10)  # mean vector 10x1
sigma = np.array(
    [
        [2.11, 0.37, -0.42, -0.2, -1.73, -0.54, 0.42, 0.81, -0.2, -0.94],
        [0.37, 1.78, -0.45, -0.25, -1.73, 0.04, 0.35, 0.38, -0.91, -0.48],
        [-0.42, -0.45, 0.87, 0.09, 0.4, -0.05, -0.25, 0.06, 0.16, 0.45],
        [-0.2, -0.25, 0.09, 0.19, 0.38, -0.04, -0.11, -0.04, 0.22, 0.17],
        [-1.73, -1.73, 0.4, 0.38, 5.3, 0.61, 0.46, 0.26, 1.74, 1.46],
        [-0.54, 0.04, -0.05, -0.04, 0.61, 0.53, 0.1, -0.3, -0.1, 0.17],
        [0.42, 0.35, -0.25, -0.11, 0.46, 0.1, 0.91, 0.51, -0.17, -0.25],
        [0.81, 0.38, 0.06, -0.04, 0.26, -0.3, 0.51, 1.55, 0.49, 0.08],
        [-0.2, -0.91, 0.16, 0.22, 1.74, -0.1, -0.17, 0.49, 1.33, 0.65],
        [-0.94, -0.48, 0.45, 0.17, 1.46, 0.17, -0.25, 0.08, 0.65, 0.88],
    ]
)  ## 10x10 covariance matrix
alpha = 1  # System-level coupling weight
c = 1.0  # Optional constant for the inequality constraint
dim = len(mu)
guess = np.zeros(dim)  ## Initial value for optimization procedure
epsilon = 0.4  # Scale used in domain transformation
# (c < 1, still consistent with the thing mentioned in the paper because we inverse the formula)
N_sobol = 1024  # number of Sobol points
m_shift = 32  # number of randomized digital shifts

In [19]:
# =========================
# Print helpers for damping
# =========================
def print_1d_history(vals, title, show=6,ind = "damping"):
    print(title)
    print("-" * len(title))
    n = len(vals)
    head = min(show, n)
    for it, d in enumerate(vals[:head]):
        print(f"iter {it:3d} | {ind} = {d:10.4g}")
    if n > show:
        print(f"... ({n - show} more iterations not shown)")

def print_2d_history(vals, title, show=6):
    print(title)
    print("-" * len(title))
    n = len(vals)
    head = min(show, n)
    for it, tup in enumerate(vals[:head]):
        tup_str = "(" + ", ".join(f"{x:10.4g}" for x in tup) + ")"
        print(f"iter {it:3d} | tuple = {tup_str}")
    if n > show:
        print(f"... ({n - show} more iterations not shown)")

## Single-level Fourier-RQMC

In [ ]:
# =========================
# Build single-level loss object, here for numerical convenience we fix the damping at the initial level
# =========================
loss_qmc_Fou_qpc_10D = RQMC_Fou_MN_qpc(
    v_mu=mu,
    v_sigma=sigma,
    N_sobol=N_sobol,
    m_shift=m_shift,
    alpha=alpha,
    c=c,
    epsilon=epsilon,
    fix_damping=True,
)


# =========================
# Internal bookkeeping helpers at each iteration
# =========================
def _set_current_m(m):
    """Store current iterate information"""
    m = np.asarray(m, dtype=float)
    loss_qmc_Fou_qpc_10D._current_m = m

    if loss_qmc_Fou_qpc_10D._pending_jac is not None:
        loss_qmc_Fou_qpc_10D._jac_components.append(loss_qmc_Fou_qpc_10D._pending_jac)
        loss_qmc_Fou_qpc_10D._pending_jac = None

    loss_qmc_Fou_qpc_10D.record_qmc_stats()


## Objective: \sum_{k=1}^{d} m_k
def objective_wrapper(m):
    m = np.asarray(m, dtype=float)
    loss_qmc_Fou_qpc_10D._current_m = m.copy()
    return loss_qmc_Fou_qpc_10D.objective(m)


## Gradient of the objective
def jac_wrapper(m):
    m = np.asarray(m, dtype=float)
    loss_qmc_Fou_qpc_10D._current_m = m.copy()
    return loss_qmc_Fou_qpc_10D.objective_jac(m)


def scipy_callback(m):
    _set_current_m(m)


# =========================
# Warm-up evaluation at initial point of the optimization
# =========================
loss_qmc_Fou_qpc_10D._current_m = np.asarray(guess, dtype=float)
loss_qmc_Fou_qpc_10D.shortfall_risk(guess)
loss_qmc_Fou_qpc_10D.shortfall_risk_jac(guess)
_set_current_m(guess)

# =========================
# Constraint E[l(X - m)] and its gradient E[∇l(X - m)]
# =========================
cons_qmc_exact = {
    "type": "ineq",
    "fun": lambda x: loss_qmc_Fou_qpc_10D.ineq_constraint(x),
    "jac": lambda x: loss_qmc_Fou_qpc_10D.ineq_constraint_jac(x),
}

# =========================
# Optimization
# =========================
res = minimize(
    fun=objective_wrapper,
    x0=guess,
    jac=jac_wrapper,
    constraints=cons_qmc_exact,
    method="SLSQP",
    options={"maxiter": maxiter, "ftol": 1e-6},
    callback=scipy_callback,
)


# =========================
# Damping histories
# =========================
print(f"Number of iterations: {res.nit}")

## Loss 1D components and its gradient
vals_1d_g = loss_qmc_Fou_qpc_10D._history_1d["g"][0]["K"]
vals_1d_f = loss_qmc_Fou_qpc_10D._history_1d["f"][0]["K"]

print_1d_history(
    vals_1d_g,
    "First 1D loss Fourier component damping by iteration",
    show=1,
)

print_1d_history(
    vals_1d_f,
    "First 1D loss gradient Fourier component damping by iteration",
    show=1,
)

## Loss 2D components and its gradient

vals_2d_h = loss_qmc_Fou_qpc_10D._history_2d["h"][(0, 1)]["K"]
vals_2d_l = loss_qmc_Fou_qpc_10D._history_2d["l"][(0, 1)]["K"]
print_2d_history(
    vals_2d_h,
    "First 2D loss Fourier component damping K by iteration",
    show=1,
)
print_2d_history(
    vals_2d_l,
    "First 2D loss gradient Fourier component damping K by iteration",
    show=1,
)
# =========================
# Final solution + relative error
# =========================
max_qmc_err = loss_qmc_Fou_qpc_10D.statistical_error_sol_RQMC(res.x)
rel_err = max_qmc_err / np.max(res.x)

print("\nFinal result")
print("------------")
print(f"Single Fourier-RQMC solution: {res.x}")
print(f"Relative statistical error : {rel_err:.6g}")

Number of iterations: 10
First 1D loss Fourier component damping by iteration
----------------------------------------------------
iter   0 | damping =      1.192
... (10 more iterations not shown)
First 1D loss gradient Fourier component damping by iteration
-------------------------------------------------------------
iter   0 | damping =     0.9736
... (10 more iterations not shown)
First 2D loss Fourier component damping K by iteration
------------------------------------------------------
iter   0 | tuple = (    0.8919,     0.9715)
... (10 more iterations not shown)
First 2D loss gradient Fourier component damping K by iteration
---------------------------------------------------------------
iter   0 | tuple = (    0.9174,     0.6602)
... (10 more iterations not shown)

Final result
------------
Single Fourier-RQMC solution: [0.34032591 0.26454636 0.25668781 0.10654063 1.30490336 0.18277428
 0.43254325 0.70704696 0.58738861 0.45773322]
Relative statistical error : 0.0848223


## Multilevel Fourier-RQMC

In [16]:
# =========================
# Build multilevel Fourier-RQMC loss object
# =========================
loss_qmc_CV_Fou_qpc_10D = RQMC_CV_Fou_MN_qpc(
    v_mu=mu,
    v_sigma=sigma,
    N_sobol=N_sobol,
    m_shift=m_shift,
    alpha=alpha,
    c=c,
    epsilon=epsilon,
)

# =========================
# Internal bookkeeping helpers at each iteration
# =========================
#
N_sobol_iter = [] # List to keep track for number of Sobol points
count = 0 # This will use to keep track when we enter the local convergence region
def _set_current_m(m):
    """Store current iterate."""
    global count
    m = np.asarray(m, dtype=float)
    loss_qmc_CV_Fou_qpc_10D._current_m = m
    
    # The commit option in this case help us to store only succesful optimization iteration
    loss_qmc_CV_Fou_qpc_10D.shortfall_risk(m, commit=True)
    loss_qmc_CV_Fou_qpc_10D.shortfall_risk_jac(m, commit=True)
    if loss_qmc_CV_Fou_qpc_10D._pending_jac is not None:
        loss_qmc_CV_Fou_qpc_10D._jac_components.append(
            loss_qmc_CV_Fou_qpc_10D._pending_jac
        )
    loss_qmc_CV_Fou_qpc_10D._pending_jac = None
    count += 1

    ## Adaptive choice of Sobol point when entering local convergence region
    N_sobol_iter.append(loss_qmc_CV_Fou_qpc_10D._N_current_loss)
    if count > 4:
        loss_qmc_CV_Fou_qpc_10D._divide_sobol(mult_factor=1/2)
        loss_qmc_CV_Fou_qpc_10D._divide_sobol(mult_factor=1/2,grad= True)
    
    loss_qmc_CV_Fou_qpc_10D.record_qmc_stats()

## Objective: \sum_{k=1}^{d} m_k
def objective_wrapper(m):
    m = np.asarray(m, dtype=float)
    loss_qmc_CV_Fou_qpc_10D._current_m = m.copy()
    return loss_qmc_CV_Fou_qpc_10D.objective(m)

## Gradient of the objective 
def jac_wrapper(m):
    m = np.asarray(m, dtype=float)
    loss_qmc_CV_Fou_qpc_10D._current_m = m.copy()
    return loss_qmc_CV_Fou_qpc_10D.objective_jac(m)

def scipy_callback(m):
    _set_current_m(m)

# =========================
# Warm-up evaluation at initial point of the optimization
# =========================
loss_qmc_CV_Fou_qpc_10D._current_m = np.asarray(guess, dtype=float)
loss_qmc_CV_Fou_qpc_10D.shortfall_risk(guess)
loss_qmc_CV_Fou_qpc_10D.shortfall_risk_jac(guess)
_set_current_m(guess)

# =========================
# Constraint E[l(X - m)] and its gradient E[∇l(X - m)]
# =========================
cons_qmc_exact = {
    "type": "ineq",
    "fun": lambda x: loss_qmc_CV_Fou_qpc_10D.ineq_constraint(x),
    "jac": lambda x: loss_qmc_CV_Fou_qpc_10D.ineq_constraint_jac(x),
}

# =========================
# Optimization
# =========================
res = minimize(
    fun=objective_wrapper,
    x0=guess,
    jac=jac_wrapper,
    constraints=cons_qmc_exact,
    method="SLSQP",
    options={"maxiter": maxiter, "ftol": 1e-6},
    callback=scipy_callback,
)


# =========================
# Damping and Sobol point histories
# =========================
print(f"Number of iterations: {res.nit}")
## Loss 1D components and its gradient
vals_1d_g = loss_qmc_CV_Fou_qpc_10D._history_1d["g"][0]["K"]
vals_1d_f = loss_qmc_CV_Fou_qpc_10D._history_1d["f"][0]["K"]

print_1d_history(
    vals_1d_g,
    "First 1D loss Fourier component damping by iteration",
    show=6,
)

print_1d_history(
    vals_1d_f,
    "First 1D loss gradient Fourier component damping by iteration",
    show=6,
)

## Loss 2D components and its gradient

vals_2d_h = loss_qmc_CV_Fou_qpc_10D._history_2d["h"][(0,1)]["K"]
vals_2d_l = loss_qmc_CV_Fou_qpc_10D._history_2d["l"][(0,1)]["K"]
print_2d_history(
    vals_2d_h,
    "First 2D loss Fourier component damping K by iteration",
    show=6,
)



Number of iterations: 10
First 1D loss Fourier component damping by iteration
----------------------------------------------------
iter   0 | damping =      1.192
iter   1 | damping =      1.251
iter   2 | damping =       1.22
iter   3 | damping =      1.196
iter   4 | damping =      1.192
iter   5 | damping =      1.192
... (5 more iterations not shown)
First 1D loss gradient Fourier component damping by iteration
-------------------------------------------------------------
iter   0 | damping =     0.9736
iter   1 | damping =      1.032
iter   2 | damping =      1.001
iter   3 | damping =     0.9763
iter   4 | damping =     0.9735
iter   5 | damping =     0.9729
... (5 more iterations not shown)
First 2D loss Fourier component damping K by iteration
------------------------------------------------------
iter   0 | tuple = (    0.8919,     0.9715)
iter   1 | tuple = (    0.9414,      1.021)
iter   2 | tuple = (    0.9155,     0.9902)
iter   3 | tuple = (    0.8947,     0.9702)
iter   

In [20]:
print_2d_history(
    vals_2d_l,
    "First 2D loss gradient Fourier component damping K by iteration",
    show=6,
)

## Sobol points
print_1d_history(N_sobol_iter,"Number of Sobol points",show = 10,ind = "N_Sobol")
# =========================
# Final solution + relative error
# =========================
max_qmc_err = loss_qmc_CV_Fou_qpc_10D.statistical_error_sol_RQMC(res.x,loc_index=5)
rel_err = max_qmc_err / np.max(res.x)

print("\nFinal result")
print("------------")
print(f"Multilevel Fourier-RQMC solution: {res.x}")
print(f"Relative statistical error : {rel_err:.6g}")

First 2D loss gradient Fourier component damping K by iteration
---------------------------------------------------------------
iter   0 | tuple = (    0.9174,     0.6602)
iter   1 | tuple = (    0.9684,     0.7086)
iter   2 | tuple = (    0.9415,     0.6783)
iter   3 | tuple = (      0.92,     0.6592)
iter   4 | tuple = (    0.9176,     0.6567)
iter   5 | tuple = (    0.9167,     0.6604)
... (5 more iterations not shown)
Number of Sobol points
----------------------
iter   0 | N_Sobol =       1024
iter   1 | N_Sobol =       1024
iter   2 | N_Sobol =       1024
iter   3 | N_Sobol =       1024
iter   4 | N_Sobol =       1024
iter   5 | N_Sobol =        512
iter   6 | N_Sobol =        256
iter   7 | N_Sobol =        128
iter   8 | N_Sobol =         64
iter   9 | N_Sobol =         32
... (1 more iterations not shown)

Final result
------------
Multilevel Fourier-RQMC solution: [0.35906786 0.27843847 0.25326074 0.10564246 1.26731467 0.18840131
 0.43285407 0.70311243 0.58827574 0.46473484]


## SAA

In [25]:
import math
import scipy.stats

MC_points = 10**5 ## Number of MC points
rv = scipy.stats.multivariate_normal(mean=mu, cov=sigma, allow_singular=True)
X = rv.rvs(size=MC_points)

# =========================
# Build SAA loss object
# =========================
loss_MC_10D = MCLossFunction(X, alpha=alpha, c=c)

def _set_current_m(m):
    """Store current iterate."""
    m = np.asarray(m, dtype=float)
    loss_MC_10D._current_m = m
    loss_MC_10D._var_iter.append(loss_MC_10D._pending_var)
    loss_MC_10D._pending_var = None

# =========================
# Internal bookkeeping helpers at each iteration
# =========================

## Objective: \sum_{k=1}^{d} m_k

def objective_wrapper(m):
    m = np.asarray(m, dtype=float)
    loss_MC_10D._current_m = m.copy()
    return loss_MC_10D.objective(m)

## Gradient of the objective 
def jac_wrapper(m):
    m = np.asarray(m, dtype=float)
    loss_MC_10D._current_m = m.copy()
    return loss_MC_10D.objective_jac(m)

def scipy_callback(m):
    _set_current_m(m)


# =========================
# Warm-up evaluation at initial point of the optimization
# =========================
loss_MC_10D._current_m = guess
loss_MC_10D.shortfall_risk(guess)
loss_MC_10D.shortfall_risk_jac(guess)
loss_MC_10D._var_iter.append(loss_MC_10D._pending_var)

# =========================
# Constraint E[l(X - m)] and its gradient E[∇l(X - m)]
# =========================
cons = {
    "type": "ineq",
    "fun": lambda x: loss_MC_10D.ineq_constraint(x),
    "jac": lambda x: loss_MC_10D.ineq_constraint_jac(x),
}

# =========================
# Optimization
# =========================
res = minimize(
    fun=objective_wrapper,
    x0=guess,
    jac=jac_wrapper,
    constraints=cons,
    method="SLSQP",
    options={"maxiter": maxiter,'ftol':1e-6},
    callback=scipy_callback,
)

# =========================
# Final solution + relative error
# =========================
print(f"Number of iterations: {res.nit}")
max_mc_err = loss_MC_10D.statistical_error_sol_MC(res.x)
rel_err = max_mc_err / np.max(res.x)

print("\nFinal result")
print("------------")

print(f"SAA solution: {res.x}")
print(f"Relative statistical error : {rel_err:.6g}")



Number of iterations: 10

Final result
------------
SAA solution: [0.33648114 0.25913846 0.26419329 0.1105696  1.32025108 0.17956281
 0.42650912 0.7146342  0.57550516 0.44665608]
Relative statistical error : 0.0635633
